# Transfer Learning en TensorFlow con Xception
### IPD434 — Computación Evolutiva e Inteligencia Artificial
**Universidad Técnica Federico Santa María**

---

El **transfer learning** (aprendizaje por transferencia) consiste en **reutilizar** un modelo ya entrenado sobre un problema grande (por ejemplo, clasificar millones de imágenes de ImageNet) como punto de partida para una tarea nueva y más pequeña. En vez de entrenar desde cero —lo que exige muchos datos y cómputo— aprovechamos las **características** (bordes, texturas, formas) que la red ya aprendió. En este notebook clasificaremos imágenes de flores (`tf_flowers`) reutilizando la arquitectura **Xception**.

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** el concepto de *transfer learning* y cuándo conviene aplicarlo.
2. **Reutilizar** un modelo preentrenado como extractor de características en TensorFlow/Keras.
3. **Realizar** *fine-tuning* de un modelo para una tarea específica.
4. **Comparar** entrenar desde cero frente a transferir conocimiento.

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb).

## 1. Setup y carga del dataset

Usaremos **TensorFlow Datasets** (`tfds`), una colección de datasets listos para usar. El dataset `tf_flowers` contiene ~3 670 fotos de flores en **5 categorías**. Lo cargamos con `as_supervised=True` para obtener pares `(imagen, etiqueta)` y lo partimos en prueba, validación y entrenamiento.

In [ ]:
# Importar TensorFlow y TensorFlow Datasets
import tensorflow_datasets as tfds
import tensorflow as tf

In [ ]:
# Cargar el dataset tf_flowers junto con su metainformacion
dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)

In [ ]:
# Partir el dataset en prueba (10%), validacion (15%) y entrenamiento (75%)
test_set, valid_set, train_set = tfds.load("tf_flowers",split=["train[0%:10%]", "train[10%:25%]", "train[25%:]"], as_supervised=True )

In [ ]:
# import tensorflow as tf

## 2. Preprocesamiento de imágenes

Cada arquitectura preentrenada espera sus entradas en un formato específico. Xception requiere imágenes de **224×224** píxeles con los valores normalizados a un rango determinado. La función `preprocess` redimensiona la imagen y aplica el preprocesamiento propio de Xception (`xception.preprocess_input`).

In [ ]:
# Previo a la utilización de nuestro dataset en la red neuronal, la red que utilizaremos provee de
# funcionalidades de pre-procesamiento ya incluídas. Por lo tanto, utilizaremos estas funcionalidades
# y las dejaremos dentro de una función
def preprocess(image, label):
    # Cambiaremos las dimensiones de la imagen de entrada
    resized_image = tf.image.resize(image, [224, 224]) # Guardamos la imagen con nuevas dimensiones 224x224
    # Luego pasamos la imagen modificada en tamaño al preprocesamiento de nuestra red. La red
    # que utilizaremos de ejemplo tiene por nombre Xception
    final_image = tf.keras.applications.xception.preprocess_input(resized_image)
    # Finalmente se retorna una imagen pre procesada (según lo indique el preprocess de xception)
    # y su etiqueta
    return final_image, label

## 3. Pipeline de datos por lotes

Aplicamos el preprocesamiento a cada partición y armamos los **lotes** (*batches*). `shuffle` mezcla el entrenamiento para que el orden no sesgue el aprendizaje, y `prefetch` solapa la preparación de datos con el cómputo del modelo.

In [ ]:
# En este apartado hacemos los batch de datos directamente desde el dataset cargado por tensorflow
batch_size = 32

# Mezclamos el dataset 
train_set = train_set.shuffle(1000)
# Tanto para training, test y validación, aplicamos la función de preprocesamiento (preprocess)
# y luego generamos los batch de datos
train_set = train_set.map(preprocess).batch(batch_size).prefetch(1)
test_set = test_set.map(preprocess).batch(batch_size).prefetch(1)
valid_set = valid_set.map(preprocess).batch(batch_size).prefetch(1)

## 4. Transfer learning con Xception

Cargamos la arquitectura **Xception** con sus pesos preentrenados en **ImageNet**. El parámetro `include_top=False` descarta la capa de salida original (1000 clases de ImageNet), dejando solo la **base convolucional** como extractor de características. Sobre ella añadimos nuestra propia salida: un `GlobalAveragePooling2D` seguido de una capa `Dense` con **5 neuronas** (una por tipo de flor) y activación `softmax`.

In [ ]:
# Acá empezamos con transfer learning. Lo que haremos será cargar una arquitectura de red neuronal desde
# tensorflow ya entrenada. Eso quiere decir que, no solo estamos cargando la arquitectura, con sus neuronas
# y conexiones, sino que también estamos cargando los PESOS DE ENTRENAMIENTO.

# Weights indica si utilizaremos pesos pre entrenados con el dataset imagenet o no
# include_top es el parámetro que indica explícitamente si quieres o no la capa de salida original
# de esta red.
# En base_model tenemos cargado nuestro modelo Xception sin la capa de salida. Nosotros podemos
# poner NUESTRA PROPIA CAPA(S) DE SALIDA
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)
# Acá agregamos nuestras capas adicionales
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(5, activation="softmax")(avg)

# Podemos conectar el input de nuestro modelo base con el output recién generado a través de 
# un modelo de Keras
model = tf.keras.Model(inputs=base_model.input, outputs=output)
# En este punto tenemos un nuevo modelo que utiliza como base la arquitectura Xception

## 5. Congelar la base (*feature extraction*)

En la modalidad de **extracción de características** congelamos los pesos de la base (`trainable = False`) para que **no** se reentrenen: solo se aprenderán las capas nuevas que agregamos. Esto es rápido y funciona bien cuando el dataset nuevo es pequeño. (Una alternativa posterior es el *fine-tuning*, que descongela parte de la base para reajustarla.)

In [ ]:
# También nosotros conversamos que es posible decidir si queremos reentrenar aquellas capas ya entrenadas

# Acá vamos capa por capa modificando el parámetro "trainable" que permite (o no) reentrenar dicha capa
for layer in base_model.layers:
    layer.trainable = False # Esto impide que las capas se re entrenen

# Con este for, sobre todas las capas de nuestro modelo, estamos impidiendo que se reentrene
# alguna de sus capas ya entrenadas

## 6. Compilación

Compilamos con `sparse_categorical_crossentropy` (etiquetas como enteros, no *one-hot*), optimizador SGD y la métrica de exactitud.

In [ ]:
# Proceso de compilación  (tal cual vimos en las clases anteriores)
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", 
              metrics=["accuracy"])

## 7. Entrenamiento

Entrenamos únicamente las capas nuevas durante 10 épocas, validando en cada una. El historial (`history`) guarda la evolución de la pérdida y la exactitud.

In [ ]:
# Entrenar el modelo (solo las capas anadidas) y guardar el historial
history = model.fit(train_set, epochs=10, validation_data=valid_set)

## 8. Predicción y evaluación

Aplicamos el modelo entrenado sobre el conjunto de prueba y comparamos las predicciones con las etiquetas reales. `model.predict` devuelve, por cada imagen, un vector de 5 probabilidades (una por clase); la clase predicha es la de mayor probabilidad.

In [ ]:
# Predecir las probabilidades de clase sobre el conjunto de prueba
model.predict(test_set)

In [ ]:
# Recorrer los lotes de prueba mostrando etiqueta real vs. prediccion
import numpy as np
for image, label in test_set:
    print("Etiqueta real",label)
    print("Predicción", model.predict(image))


## 📌 Resumen

| Concepto | Idea clave |
|----------|-----------|
| Transfer learning | Reutilizar un modelo preentrenado en una tarea nueva |
| Feature extraction | Congelar la base y entrenar solo la cabeza nueva |
| Fine-tuning | Descongelar (parte de) la base y reajustar sus pesos |
| `include_top=False` | Cargar la base sin la capa de salida original |
| `preprocess_input` | Adaptar las imágenes al formato que espera el modelo base |

**Idea central:** entrenar desde cero exige muchos datos y cómputo; el transfer learning reutiliza el conocimiento de ImageNet y alcanza buena exactitud con un dataset pequeño como `tf_flowers`.

## 📚 Referencias

- **Chollet, F. (2017).** *Xception: Deep Learning with Depthwise Separable Convolutions*. CVPR. https://arxiv.org/abs/1610.02357
- **TensorFlow.** *Transfer learning and fine-tuning* (guía oficial). https://www.tensorflow.org/tutorials/images/transfer_learning
- **Keras.** Documentación de `tf.keras.applications`. https://keras.io/api/applications/
- **TensorFlow Datasets.** Catálogo — `tf_flowers`. https://www.tensorflow.org/datasets/catalog/tf_flowers